# 08. Evaluation Metrics

Evaluate Popularity, Rating-based, Content-based, and Collaborative Filtering models on the test set.

In [1]:
import pandas as pd
import numpy as np
import pickle
import os
from sklearn.metrics.pairwise import cosine_similarity
from scipy.sparse import load_npz
from collections import defaultdict
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt
import seaborn as sns

# Set plotting style
sns.set_theme(style='whitegrid')


## 1. Load Data & Models

In [2]:
test_df = pd.read_csv('test_interactions.csv')
print(f'Test shape: {test_df.shape}')

# Get ground truth for each user
# Only consider items with rating >= 4 as positive interactions (hits)
positive_test = test_df[test_df['rating'] >= 4]
ground_truth = positive_test.groupby('authorid')['recipeid'].apply(set).to_dict()

# We will only evaluate users who have at least one positive interaction in test
eval_users = list(ground_truth.keys())
print(f'Total users to evaluate: {len(eval_users)}')

Test shape: (166906, 10)
Total users to evaluate: 26116


In [3]:
# Optional: Sample a subset of users to speed up evaluation
import random
random.seed(42)
if len(eval_users) > 5000:
    eval_users = random.sample(eval_users, 5000)
    print(f'Sampled {len(eval_users)} users for faster evaluation.')

Sampled 5000 users for faster evaluation.


In [4]:
# Load Baselines
with open('output/pop_recs.pkl', 'rb') as f:
    pop_recs = pickle.load(f)
    
with open('output/rating_recs.pkl', 'rb') as f:
    rating_recs = pickle.load(f)

# Load Content-based
with open('output/content_items.pkl', 'rb') as f:
    content_items = pickle.load(f)
content_tfidf = load_npz('output/tfidf_matrix.npz')

# Map item_id to index
item2idx = {item: idx for idx, item in enumerate(content_items)}
idx2item = {idx: item for idx, item in enumerate(content_items)}

# For Content-based, we need train_df to know what a user liked in the past
train_df = pd.read_csv('train_interactions.csv')
# Filter train_df for eval users
user_train_history = train_df[train_df['authorid'].isin(eval_users)].groupby('authorid')['recipeid'].apply(list).to_dict()

# Load SVD
with open('output/svd_model.pkl', 'rb') as f:
    svd_model = pickle.load(f)

## 2. Evaluation Metrics Definitions

In [5]:
def precision_at_k(actual, predicted, k):
    if not actual:
        return 0.0
    pred_k = set(predicted[:k])
    hits = len(pred_k.intersection(actual))
    return hits / k

def recall_at_k(actual, predicted, k):
    if not actual:
        return 0.0
    pred_k = set(predicted[:k])
    hits = len(pred_k.intersection(actual))
    return hits / len(actual)

def hitrate_at_k(actual, predicted, k):
    if not actual:
        return 0.0
    pred_k = set(predicted[:k])
    return 1.0 if len(pred_k.intersection(actual)) > 0 else 0.0

def ndcg_at_k(actual, predicted, k):
    if not actual:
        return 0.0
    dcg = 0.0
    for i, p in enumerate(predicted[:k]):
        if p in actual:
            dcg += 1.0 / np.log2(i + 2)
    
    idcg = sum((1.0 / np.log2(i + 2)) for i in range(min(k, len(actual))))
    return dcg / idcg if idcg > 0 else 0.0

## 3. Generate Predictions & Compute Metrics

In [6]:
metrics = {'Model': [], 'Precision@5': [], 'Recall@5': [], 'HitRate@10': [], 'NDCG@10': []}

def evaluate_model(model_name, predictions_dict):
    p5, r5, hr10, ndcg10 = [], [], [], []
    for u in eval_users:
        actual = ground_truth.get(u, set())
        if not actual:
            continue
        preds = predictions_dict.get(u, [])
        
        p5.append(precision_at_k(actual, preds, 5))
        r5.append(recall_at_k(actual, preds, 5))
        hr10.append(hitrate_at_k(actual, preds, 10))
        ndcg10.append(ndcg_at_k(actual, preds, 10))
        
    metrics['Model'].append(model_name)
    metrics['Precision@5'].append(np.mean(p5))
    metrics['Recall@5'].append(np.mean(r5))
    metrics['HitRate@10'].append(np.mean(hr10))
    metrics['NDCG@10'].append(np.mean(ndcg10))
    print(f"{model_name} Evaluation Complete.")

### Popularity and Rating-Based

In [7]:
# Since these are non-personalized, all users get the same top K
pop_preds = {u: pop_recs[:10] for u in eval_users}
evaluate_model('Popularity', pop_preds)

rating_preds = {u: rating_recs[:10] for u in eval_users}
evaluate_model('Rating-based', rating_preds)

Popularity Evaluation Complete.
Rating-based Evaluation Complete.


### Content-Based

In [8]:
cb_preds = {}
# Only consider predicting from the subset of content items we have features for
candidate_items = set(content_items)

for u in tqdm(eval_users, desc='Evaluating Content-based'):
    history = user_train_history.get(u, [])
    history = [str(h) for h in history if str(h) in item2idx]
    
    if not history:
        # Fallback to popularity
        cb_preds[u] = pop_recs[:10]
        continue
        
    # Get indices of history items
    hist_idx = [item2idx[h] for h in history]
    # Get user profile by averaging tfidf vectors
    user_profile = np.asarray(content_tfidf[hist_idx].mean(axis=0))
    
    # Compute similarity to all items
    sims = cosine_similarity(user_profile, content_tfidf).flatten()
    
    # Sort indices by descending similarity
    sim_indices = sims.argsort()[::-1]
    
    # Extract top 10 removing already interacted items
    history_set = set(history)
    top_preds = []
    for idx in sim_indices:
        item = idx2item[idx]
        if item not in history_set:
            top_preds.append(int(item))
        if len(top_preds) == 10:
            break
    cb_preds[u] = top_preds

evaluate_model('Content-based', cb_preds)

Evaluating Content-based:   0%|          | 0/5000 [00:00<?, ?it/s]

Content-based Evaluation Complete.


### Collaborative Filtering (SVD)

In [9]:
# Get all unique items in trainset
all_items = train_df['recipeid'].unique().tolist()

svd_preds = {}
for u in tqdm(eval_users, desc='Evaluating SVD'):
    history_set = set(user_train_history.get(u, []))
    
    # Score a sample of 500 items to speed things up (in production you'd score more or use nearest neighbors)
    # To make it fair but fast, score 500 random items not in history + top popularity items
    candidates = list(set(pop_recs[:500]) | set(random.sample(all_items, min(500, len(all_items)))))
    candidates = [c for c in candidates if c not in history_set]
    
    preds = []
    for i in candidates:
        pred = svd_model.predict(u, i).est
        preds.append((i, pred))
        
    # Sort by predicted rating
    preds.sort(key=lambda x: x[1], reverse=True)
    svd_preds[u] = [p[0] for p in preds[:10]]

evaluate_model('Collaborative filtering', svd_preds)

Evaluating SVD:   0%|          | 0/5000 [00:00<?, ?it/s]

Collaborative filtering Evaluation Complete.


## 4. Final Results Table

In [10]:
results_df = pd.DataFrame(metrics)
display(results_df)

,Model,Precision@5,Recall@5,HitRate@10,NDCG@10
0,Popularity,0.00776,0.011431,0.0618,0.013920
1,Rating-based,0.00056,0.000334,0.0058,0.000884
2,Content-based,0.00224,0.005210,0.0184,0.005381
3,Collaborative filtering,0.00156,0.001858,0.0150,0.002524
